In [3]:
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from config import GOOGLE_AI_API_KEY
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import ToolMessage

CHAT_MODEL = 'gemma-4-31b-it'
# CHAT_MODEL = 'gemini-3.1-flash-lite'

model = init_chat_model(CHAT_MODEL, model_provider='google_genai', api_key=GOOGLE_AI_API_KEY)

In [11]:
@tool
def estimate_delivery_days(region: str, delayed: bool = False) -> str:
    """배송 지역과 지연 여부로 예상 배송일을 계산한다."""
    base_days = {"서울": 1, "수도권": 2, "지방": 3, "제주": 5}.get(region, 3)
    if delayed:
        base_days += 2
    return f"예상 배송일: {base_days}일"


@tool
def calculate_delay_compensation(delay_days: int, order_amount: int) -> str:
    """배송 지연 일수와 주문 금액으로 보상 쿠폰 금액을 계산한다."""
    if delay_days < 3:
        return "보상 대상 아님"
    coupon = min(int(order_amount * 0.1), 10000)
    return f"배송 지연 {delay_days}일: {coupon}원 쿠폰 권장"

# LLM이 사용할 수 있는 도구 목록
llm_tools = model.bind_tools([estimate_delivery_days, calculate_delay_compensation])

# 도구 이름으로 실제 도구 함수를 찾기 위한 딕셔너리.
tools_by_name = {t.name: t for t in [estimate_delivery_days, calculate_delay_compensation]}

def call_llm(state: MessagesState) -> dict:
    """LLM 한 번 호출. 도구를 부를 수도, 답을 줄 수도."""
    ai = llm_tools.invoke(state["messages"])
    return {"messages": [ai]}


def call_tools(state: MessagesState) -> dict:
    """LLM 이 부른 도구들을 실행하고 ToolMessage 로 결과 반환."""
    last = state["messages"][-1]
    tool_msgs = []
    for call in last.tool_calls:
        tool = tools_by_name[call["name"]]
        result = tool.invoke(call["args"])
        tool_msgs.append(ToolMessage(str(result), tool_call_id=call["id"]))
    return {"messages": tool_msgs}


def needs_tools(state: MessagesState) -> str:
    """마지막 ai 메시지에 tool_calls 가 있으면 'tools', 없으면 'end'."""
    return "tools" if state["messages"][-1].tool_calls else "end"

graph2 = StateGraph(MessagesState)
graph2.add_node('llm', call_llm)
graph2.add_node('tools', call_tools)

graph2.add_edge(START, 'llm')
graph2.add_conditional_edges(
    "llm",                          # llm 노드 끝나면
    needs_tools,                    # 라우터로 결정
    {"tools": "tools", "end":END}
)
graph2.add_edge("tools", "llm")     # 도구 실행 후 다시 llm으로 (루프!!)

app2 = graph2.compile()

result = app2.invoke({
    "messages":[HumanMessage("제주 지역 고객 주문이 5일 지연됐어. 주문금액 80000원인데 예상 배송일과 보상 쿠폰을 계산해줘.")]
})
print(result['messages'][-1].content)
for m in result['messages']:
    print(f"[{type(m).__name__}] {str(m.content)} ")


NameError: name 'ToolMessage' is not defined

KeyboardInterrupt: 

In [4]:
tool_node = ToolNode([estimate_delivery_days, calculate_delay_compensation])

graph_pb = StateGraph(MessagesState)
graph_pb.add_node('llm', call_llm)
graph_pb.add_node('tools', tool_node)
graph_pb.add_edge(START, 'llm')
graph_pb.add_conditional_edges('llm', tools_condition)
graph_pb.add_edge('tools', 'llm')
app_pd = graph_pb.compile()

result = app_pd.invoke(
    {"messages":[HumanMessage("제주 배송 지연 5일, 주문금액 80000원인 티켓의 예상 배송일과 보상 쿠폰을 알려줘.")]},
    config={'recursion_limit':10}
)
print(result)

{'messages': [HumanMessage(content='제주 배송 지연 5일, 주문금액 80000원인 티켓의 예상 배송일과 보상 쿠폰을 알려줘.', additional_kwargs={}, response_metadata={}, id='04661436-0ea9-4728-b3ba-6f1b9c42bb42'), AIMessage(content=[{'type': 'thinking', 'thinking': 'The user is asking for two things:\n1. The estimated delivery date for a ticket sent to Jeju with a delay.\n2. The compensation coupon amount for a 5-day delay on an order of 80,000 KRW.\n\nI need to call two tools:\n1. `estimate_delivery_days` to get the estimated delivery date.\n   - `region`: "제주" (Jeju)\n   - `delayed`: true (since the user mentioned "배송 지연")\n2. `calculate_delay_compensation` to get the compensation amount.\n   - `delay_days`: 5\n   - `order_amount`: 80000\n\nI will call these tools.'}], additional_kwargs={'function_call': {'name': 'calculate_delay_compensation', 'arguments': '{"order_amount": 80000, "delay_days": 5}'}, '__gemini_function_call_thought_signatures__': {'gnc5mb3t': 'EiYKJGUyNDgzMGE3LTVjZDYtNDJmZS05OThiLWVlNTM5ZTcyYjljMw=='}},

In [5]:
from typing import Literal

class RouteDecision(BaseModel):
    """사용자 의도 분류."""
    intent: Literal["support_ticket", "code_help", "casual_chat"] = Field(
        description="사용자가 원하는 것의 유형"
    )

router_llm = model.with_structured_output(RouteDecision)


def llm_router(state: MessagesState) -> str:
    """LLM 으로 의도 분류 후 다음 노드 결정."""
    decision = router_llm.invoke([
        HumanMessage("아래 메시지의 의도를 분류해줘:\n" + state["messages"][-1].content)
    ])
    return decision.intent


def answer_support(state: MessagesState) -> dict:
    msgs = state["messages"] + [HumanMessage("너는 이커머스 고객지원 매니저. 티켓 처리 관점으로 답해줘.")]
    return {"messages": [model.invoke(msgs)]}


def answer_code(state: MessagesState) -> dict:
    msgs = state["messages"] + [HumanMessage("너는 시니어 개발자. 코드 위주로 답해.")]
    return {"messages": [model.invoke(msgs)]}


def answer_casual(state: MessagesState) -> dict:
    msgs = state["messages"] + [HumanMessage("친구처럼 짧고 가볍게 답해.")]
    return {"messages": [model.invoke(msgs)]}

graph3 = StateGraph(MessagesState)
graph3.add_node("support", answer_support)
graph3.add_node("code", answer_code)
graph3.add_node("casual", answer_casual)
graph3.add_conditional_edges(
    START, 
    llm_router,
    {"support_ticket": "support",
     "code_help" : "code",
     "casual_chat" : "casual"
})
graph3.add_edge("support",END)
graph3.add_edge("code",END)
graph3.add_edge("casual",END)
app3 = graph3.compile()

questions =[
    "고객 배송이 5일째 안 온다고 화났어. 어떻게 응대할까?",
    "for문에서 enumerate가 뭐야?",
    "오늘 점심 뭐먹?"
]

for q in questions:
    print(f"Q: {q}")
    r = app3.invoke({"messages": [HumanMessage(q)]})
    print(f"A: {r['messages'][-1].content[:80]}")
    print()

Q: 고객 배송이 5일째 안 온다고 화났어. 어떻게 응대할까?
A: [{'type': 'thinking', 'thinking': '\n*   *Scenario:* A customer is angry because their delivery hasn\'t arrived for 5 days.\n*   *Role:* E-commerce Customer Support (CS) Manager.\n*   *Perspective:* Ticket handling (process-oriented, structured).\n*   *Goal:* Provide a professional, empathetic, and effective response strategy and actual templates.\n\n    *   *Step 1: Acknowledgment & Empathy.* The customer is angry. The first step is to lower the tension.\n    *   *Step 2: Investigation (The "Why").* Why is it late? (Courier issue, stock issue, wrong address, lost package).\n    *   *Step 3: Solution/Alternative.* How do we fix it? (Expedite, resend, refund, compensation).\n    *   *Step 4: Closure & Follow-up.* Ensure the customer is satisfied and the ticket is closed properly.\n\n    *   *Phase 1: Immediate Response (Triage).*\n        *   Goal: Stop the bleeding. Acknowledge the delay.\n        *   Key phrase: "We apologize for the inconvenience

In [ ]:
from config import LANGSMITH_PROJECT
def stage_summary(state: MessagesState) -> dict:
    """1단계: 고객 문의를 한 문장으로 요약."""
    msgs = state["messages"] + [HumanMessage("이 고객 문의를 한 문장으로 요약해줘.")]
    return {"messages": [model.invoke(msgs)]}


def stage_classify(state: MessagesState) -> dict:
    """2단계: 요약을 보고 티켓 유형 분류."""
    msgs = state["messages"] + [HumanMessage("위 내용을 배송/환불/교환/계정 중 하나의 티켓 유형으로 분류하고 이유를 한 줄로 써줘.")]
    return {"messages": [model.invoke(msgs)]}


def stage_reply(state: MessagesState) -> dict:
    """3단계: 고객에게 보낼 답변 초안 작성."""
    msgs = state["messages"] + [HumanMessage("이 티켓에 대해 고객에게 보낼 답변 초안을 정중하게 한 문단으로 작성해줘.")]
    return {"messages": [model.invoke(msgs)]}

graph = StateGraph(MessagesState)
graph.add_node("summary", stage_summary)
graph.add_node("classify", stage_classify)
graph.add_node("reply", stage_reply)
graph.add_edge(START, "summary")
graph.add_edge("summary", "classify")
graph.add_edge("classify", "reply")
graph.add_edge("reply", END)
app = graph.compile()

result = app.invoke({"messages":[HumanMessage("주문 O-1001이 5일째 배송되지 않았고 고객이 환불 가능 여부를 문의했습니다.")]})
print("최종 고객 답변:", result["messages"][-1].content)
print()
print(f"LangSmith 에서 'summary -> classify -> reply' 트리 확인:")
print(f"  https://smith.langchain.com/projects/{LANGSMITH_PROJECT}")

from langchain_core.runnables import RunnableConfig

config = RunnableConfig(
    tags=['day2', 'support-ticket-graph'],
    metadata={'scenario': "delayed_delivery", "version":"v0.1"}
)

result = app.invoke(
    {"messages":[HumanMessage("고객이 반품 신청 후 환불이 아직 안 들어왔다고 문의했습니다.")]},
    config=config
)
result['messages'][-1].content


최종 고객 답변: [{'type': 'thinking', 'thinking': "Order O-1001.\nDelivery delayed for 5 days.\nCustomer is asking if a refund is possible.\nWrite a polite draft response to the customer in one paragraph.\n\n    *   Apologize for the delay.\n    *   Acknowledge the specific order (O-1001).\n    *   Address the refund request (either confirm it's possible or state that you're checking/processing it).\n    *   Provide a clear next step or solution.\n    *   Maintain a professional and empathetic tone.\n\n    *   *Option 1 (Direct Refund Approval):* Sorry for the delay on O-1001. Yes, you can get a refund. We will process it now.\n    *   *Option 2 (Checking status first, then offering refund):* Sorry for the delay on O-1001. We are checking where it is. If you want a refund, we can do that.\n    *   *Option 3 (Empathetic and Action-oriented - Best for CS):* Deeply apologize for the 5-day delay. We understand the frustration. We can process the refund immediately or try to speed up delivery.\n\

In [5]:
from langgraph.types import Command
from typing import Literal

# 1. 평가 노드 (Command를 통해 동적으로 pass_node 또는 fail_node로 분기)
def grade_answer(state: MessagesState) -> Command[Literal["pass_node", "fail_node"]]:
    """마지막 메시지에 주문번호나 처리 가능 키워드가 있으면 pass_node, 없으면 fail_node로 이동."""
    last_message = state["messages"][-1].content.lower()
    is_pass = "o-" in last_message or "가능" in last_message or "확인" in last_message

    # 상태 업데이트할 값 설정 (여기서는 생략하고 이동만 처리)
    return Command(
        # 다음 노드를 함수 안에서 직접 지정
        goto='pass_node' if is_pass else "fail_node",
        # 동시에 state도 업데이터(선택)
        update={'messages': [HumanMessage(f"[검토 결과]{'처리 가능' if is_pass else '추가 정보 필요'}")]}
    )

# 2. 타겟 노드 정의
def pass_node(state: MessagesState) -> dict:
    return {"messages": [HumanMessage("주문 정보가 확인되었습니다. 다음 단계로 이동합니다.")]}

def fail_node(state: MessagesState) -> dict:
    return {"messages": [HumanMessage("주문번호나 문의 유형을 먼저 확인해야 합니다.")]}

# 3. 그래프 정의
workflow = StateGraph(MessagesState)
workflow.add_node("grade", grade_answer) # 라우터 역할과 평가 역할을 동시에 하는 노드
workflow.add_node("pass_node", pass_node)
workflow.add_node("fail_node", fail_node)

# START에서 바로 grade 노드로 진입
workflow.add_edge(START, "grade")
workflow.add_edge("pass_node", END)
workflow.add_edge("fail_node", END)

app = workflow.compile()

# 처리 가능 시나리오
result1 = app.invoke({"messages": [HumanMessage("주문번호 O-1001 확인 가능합니다.")]})
for m in result1["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

print()

# 추가 정보 필요 시나리오
result2 = app.invoke({"messages": [HumanMessage("배송이 안 왔어요.")]})
for m in result2["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")


  [HumanMessage] 주문번호 O-1001 확인 가능합니다.
  [HumanMessage] [검토 결과]처리 가능
  [HumanMessage] 주문 정보가 확인되었습니다. 다음 단계로 이동합니다.

  [HumanMessage] 배송이 안 왔어요.
  [HumanMessage] [검토 결과]추가 정보 필요
  [HumanMessage] 주문번호나 문의 유형을 먼저 확인해야 합니다.


In [7]:
from typing import TypedDict

class TicketFlowState(TypedDict):
    messages: Annotated[list, add_messages]
    stage: int                      # 0 -> 1 -> 2 -> 3으로 진행

def stage_start(state: TicketFlowState) -> Command[Literal['stage_policy_check', 'stage_order_check']]:
    """처리 시작 후, 짝수 stage 면 주문 확인으로, 홀수면 정책 확인으로."""
    next_stage = state["stage"] + 1
    goto = "stage_order_check" if next_stage % 2 == 0 else "stage_policy_check"
    return Command(
        goto=goto,
        update={"stage": next_stage, "messages":[HumanMessage(f"[stage {next_stage}] 처리 시작")]}
    )

def stage_order_check(state: TicketFlowState) -> Command[Literal['stage_start', "__end__"]]:
    """주문/배송 상태 확인. stage 3 이상이면 답변 발송 후 종료."""
    if state["stage"] >= 3:
        return Command(
            #goto=END,
            goto="__end__",
            update={"messages": [HumanMessage("고객 답변 발송 완료. 티켓 종료.")]})
    return Command(
        goto="stage_start",
        update={"messages":[HumanMessage(f"[stage {state['stage']}]  주문/배송 상태 확인 완료")]}
    )

def stage_policy_check(state: TicketFlowState) -> Command[Literal['stage_start', "__end__"]]:
    """환불/보상 정책 확인 후 다음 단계로."""
    return Command(
        goto="stage_start",
        update={"messages":[HumanMessage(f"[stage {state['stage']}]  환불/보상 상태 확인 완료")]}
    )

# 3. 그래프 정의
workflow = StateGraph(TicketFlowState)
workflow.add_node("stage_start", stage_start) # 라우터 역할과 평가 역할을 동시에 하는 노드
workflow.add_node("stage_order_check", stage_order_check)
workflow.add_node("stage_policy_check", stage_policy_check)

# START에서 바로 grade 노드로 진입
workflow.add_edge(START, "stage_start")
workflow.add_edge("stage_order_check", END)
workflow.add_edge("stage_policy_check", END)

app = workflow.compile()

# 처리 가능 시나리오
result1 = app.invoke({'stage':0,"messages": [HumanMessage("주문번호 O-1001 확인 가능합니다.")]})
for m in result1["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

print()

# 추가 정보 필요 시나리오
result2 = app.invoke({'stage':5, "messages": [HumanMessage("배송이 안 왔어요.")]})
for m in result2["messages"]:
    print(f"  [{type(m).__name__}] {m.content}")

  [HumanMessage] 주문번호 O-1001 확인 가능합니다.
  [HumanMessage] [stage 1] 처리 시작
  [HumanMessage] [stage 1]  환불/보상 상태 확인 완료
  [HumanMessage] [stage 2] 처리 시작
  [HumanMessage] [stage 2]  주문/배송 상태 확인 완료
  [HumanMessage] [stage 3] 처리 시작
  [HumanMessage] [stage 3]  환불/보상 상태 확인 완료
  [HumanMessage] [stage 4] 처리 시작
  [HumanMessage] 고객 답변 발송 완료. 티켓 종료.

  [HumanMessage] 배송이 안 왔어요.
  [HumanMessage] [stage 6] 처리 시작
  [HumanMessage] 고객 답변 발송 완료. 티켓 종료.


In [2]:
@tool
def lookup_order_status(order_id: str) -> str:
    """주문번호로 배송/처리 상태를 조회한다 (데모 가짜 DB)."""
    db = {
        "O-1001": "배송 지연 5일. 제주 물류센터에서 대기 중.",
        "O-1002": "배송 완료. 고객 수령 확인됨.",
        "O-1003": "반품 접수 완료. 환불 심사 대기 중.",
    }
    return db.get(order_id, f"{order_id} 주문 정보 없음")


@tool
def lookup_refund_policy(product_type: str) -> str:
    """상품 유형으로 환불 정책을 조회한다."""
    table = {
        "전자제품": "미개봉 7일 이내 환불 가능. 배송 지연은 보상 대상.",
        "의류": "착용 흔적이 없으면 7일 이내 환불 가능.",
        "식품": "신선식품은 단순 변심 환불 불가. 파손/오배송만 처리.",
    }
    return table.get(product_type, "정책 정보 없음")

tools = [lookup_order_status, lookup_refund_policy]
llm_tools = model.bind_tools(tools)

# 노드 1: LLM 호출 (Reason + Act)
def call_llm(state : MessagesState) -> dict:
    ai = llm_tools.invoke(state['messages'])
    return {"messages":[ai]}

# 노드 2: 도구 실행 (Observe)
tool_node = ToolNode(tools)

graph = StateGraph(MessagesState)
graph.add_node('llm', call_llm)
graph.add_node('tools', tool_node)
graph.add_edge(START, 'llm')
graph.add_conditional_edges('llm', tools_condition)
graph.add_edge('tools', 'llm')
app = graph.compile()

# 실행: 두 도구가 모두 필요한 질문
result = app.invoke(
    {"messages": [HumanMessage("O-1001 주문 상태와 전자제품 환불 정책을 확인해서 고객에게 답할 내용을 알려줘.")]},
    config={"recursion_limit": 10},
)


print("=== 전체 흐름 ===")
for m in result["messages"]:
    print(f"  [{type(m).__name__:14}] {str(m.content)[:90]}")

print()
print("최종 답:", result["messages"][-1].content)

=== 전체 흐름 ===
  [HumanMessage  ] O-1001 주문 상태와 전자제품 환불 정책을 확인해서 고객에게 답할 내용을 알려줘.
  [AIMessage     ] [{'type': 'thinking', 'thinking': 'The user wants to know the status of order "O-1001" and
  [ToolMessage   ] 배송 지연 5일. 제주 물류센터에서 대기 중.
  [ToolMessage   ] 미개봉 7일 이내 환불 가능. 배송 지연은 보상 대상.
  [AIMessage     ] [{'type': 'thinking', 'thinking': 'The order status for O-1001 is "Delivery delayed by 5 d

최종 답: [{'type': 'thinking', 'thinking': 'The order status for O-1001 is "Delivery delayed by 5 days. Waiting at Jeju logistics center."\nThe refund policy for electronics is "Refund possible within 7 days if unopened. Delivery delays are eligible for compensation."\n\nI should combine these two pieces of information to suggest a response to the customer.\nThe customer is likely frustrated by the delay. I should apologize for the delay, explain the current status, and mention that since it\'s a delivery delay, it is eligible for compensation according to the policy. I should also mention the gener

In [4]:
@tool
def lookup_order_status(order_id: str) -> str:
    """주문번호로 배송/처리 상태를 조회한다."""
    db = {
        "O-1001": "배송 지연 5일. 제주 물류센터에서 대기 중.",
        "O-1002": "배송 준비 중. 내일 출고 예정.",
        "O-1003": "배송 완료. 고객 수령 확인됨.",
        "O-1004": "반품 접수 완료. 환불 심사 대기 중.",
    }
    return db.get(order_id, f"{order_id} 주문 정보 없음")


@tool
def calculate_refund_amount(order_amount: int, used_coupon: int = 0, opened: bool = False) -> str:
    """주문금액, 사용 쿠폰, 개봉 여부로 환불 가능 금액을 계산한다."""
    if opened:
        return "개봉 상품: 단순 변심 환불 불가. 하자 여부 확인 필요"
    refund = max(order_amount - used_coupon, 0)
    return f"환불 가능 금액: {refund}원"


@tool
def calculate_delay_compensation(delay_days: int, order_amount: int) -> str:
    """배송 지연 일수와 주문 금액으로 보상 쿠폰 금액을 계산한다."""
    if delay_days < 3:
        return "보상 대상 아님"
    coupon = min(int(order_amount * 0.1), 10000)
    return f"배송 지연 보상 쿠폰: {coupon}원"


@tool
def search_policy(keyword: str) -> str:
    """고객지원 정책 정보 검색."""
    policies = {
        "배송 지연": "출고 예정일보다 3일 이상 지연되면 주문금액의 10%, 최대 1만원 쿠폰 보상.",
        "전자제품": "미개봉 전자제품은 수령 후 7일 이내 환불 가능. 개봉 상품은 하자 확인 필요.",
        "식품": "신선식품은 단순 변심 환불 불가. 파손/오배송은 사진 확인 후 처리.",
        "쿠폰": "주문 취소 시 사용 쿠폰은 유효기간 내 자동 복구. 일부 프로모션 쿠폰 제외.",
    }
    for k, v in policies.items():
        if k in keyword:
            return v
    return "관련 정책 정보 없음"


@tool
def draft_reply_template(issue_type: str) -> str:
    """문의 유형(배송 지연/환불/교환)에 맞는 고객 답변 템플릿을 제안한다."""
    templates = {
        "배송 지연": "불편을 드려 죄송합니다. 현재 배송 상태를 확인했으며 지연 사유와 예상 일정을 안내드리겠습니다.",
        "환불": "환불 가능 여부와 예상 환불 금액을 확인해 안내드립니다. 결제수단별 처리 기간도 함께 확인해드리겠습니다.",
        "교환": "교환 가능 조건을 확인한 뒤 회수 접수와 재출고 일정을 안내드리겠습니다.",
    }
    return templates.get(issue_type, "문의 내용을 확인한 뒤 필요한 조치와 예상 일정을 안내드리겠습니다.")


tools= [
    lookup_order_status,
    calculate_refund_amount,
    calculate_delay_compensation,
    search_policy,
    draft_reply_template,
]

llm_tools = model.bind_tools(tools)
def call_llm(state: MessagesState) -> dict:
    # 시스템 프롬프트로 일관된 톤 유지
    msgs = [SystemMessage("너는 이커머스 고객지원 티켓 처리를 돕는 한국어 비서. 필요하면 도구를 적극적으로 활용해.")] + state["messages"]
    return {"messages": [llm_tools.invoke(msgs)]}

graph = StateGraph(MessagesState)
graph.add_node('llm', call_llm)
graph.add_node('tools', ToolNode(tools))
graph.add_edge(START, "llm")
graph.add_conditional_edges("llm", tools_condition)
graph.add_edge("tools", "llm")
app = graph.compile()

q = "O-1001 주문 상태를 확인하고, 주문금액 120000원에 쿠폰 10000원을 쓴 미개봉 전자제품의 환불 가능 금액과 배송 지연 5일 보상 쿠폰을 계산해줘."

result = app.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": 15})
print("Q:", q)
print()
print("=== 도구 호출 흐름 ===")
for m in result["messages"]:
    name = type(m).__name__
    if name == "AIMessage" and m.tool_calls:
        for call in m.tool_calls:
            print(f"  [Act]     {call['name']}({call['args']})")
    elif name == "ToolMessage":
        print(f"  [Observe] {m.content}")
print()
print("최종 답:", result["messages"][-1].content)

q = "전자제품 환불 정책과 배송 지연 답변 템플릿을 참고해서 고객에게 보낼 답변 초안을 작성해줘."

result = app.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": 15})
print("최종 답:", result["messages"][-1].content)

q = "O-1001, O-1002, O-1003 세 주문의 상태를 각각 확인해서 우선 처리 순서를 정해줘."

result = app.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": 15})
print("최종 답:", result["messages"][-1].content)

Q: O-1001 주문 상태를 확인하고, 주문금액 120000원에 쿠폰 10000원을 쓴 미개봉 전자제품의 환불 가능 금액과 배송 지연 5일 보상 쿠폰을 계산해줘.

=== 도구 호출 흐름 ===
  [Act]     lookup_order_status({'order_id': 'O-1001'})
  [Act]     calculate_refund_amount({'opened': False, 'order_amount': 120000, 'used_coupon': 10000})
  [Act]     calculate_delay_compensation({'delay_days': 5, 'order_amount': 120000})
  [Observe] 배송 지연 5일. 제주 물류센터에서 대기 중.
  [Observe] 환불 가능 금액: 110000원
  [Observe] 배송 지연 보상 쿠폰: 10000원

최종 답: 요청하신 내용을 확인한 결과입니다.

1. **주문 상태 (O-1001):** 현재 제주 물류센터에서 대기 중이며, 배송이 5일 지연되고 있습니다.
2. **환불 가능 금액:** 미개봉 전자제품 기준으로 환불 가능 금액은 **110,000원**입니다.
3. **배송 지연 보상:** 5일 지연에 따른 보상 쿠폰 금액은 **10,000원**으로 계산되었습니다.
최종 답: [{'type': 'thinking', 'thinking': 'I have obtained the refund policy for electronics ("Unopened electronics can be refunded within 7 days of receipt. Opened products require a defect check.") and the response template for delivery delays ("We apologize for the inconvenience. We have checked the delivery status and will inform you of 